# 02 · Model evaluation

Tests every trained model on the **same 20% test split used during training** (`random_state=42`), so these numbers come from data the models never saw.

**Before running:** run `retrain_models.bat` so the cleaned datasets and freshly trained models exist.

In [ ]:
import sys
from pathlib import Path

# Jupyter starts inside notebooks/, so the project root is one level up.
PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report
from sklearn.model_selection import train_test_split

from services import matching_model, model, screening_model
from services.parser import extract_text_from_bytes
from services.recommender import recommend_as_dict

## 1. Role classifier (TF-IDF + Linear SVM)

*Precision* = when it says "Teacher", how often is it right. *Recall* = of all real teachers, how many it found. *F1* = the balance of both.

In [ ]:
texts, categories = model.load_resume_training_data()
role_model, label_encoder = model.load_role_classifier()
encoded = label_encoder.transform(categories)
_, x_test, _, y_test = train_test_split(texts, encoded, test_size=0.2, random_state=42, stratify=encoded)

role_predictions = role_model.predict(x_test)
print(classification_report(y_test, role_predictions, target_names=label_encoder.classes_, zero_division=0))

Confusion matrix: the **diagonal** is correct predictions; anything off the diagonal is one category mistaken for another.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay.from_predictions(
    label_encoder.inverse_transform(y_test),
    label_encoder.inverse_transform(role_predictions),
    xticks_rotation="vertical",
    ax=ax,
    colorbar=False,
)
ax.set_title("Role classifier - confusion matrix (test set)")
plt.show()

## 2. Resume–job matching model (character TF-IDF pair features + Logistic Regression)

The ROC curve shows how well the model separates matches from non-matches at every threshold. The area under it (AUC) is 0.5 for random guessing and 1.0 for perfect.

In [ ]:
resume_texts, job_texts, labels = matching_model.load_matching_training_data()
vectorizer, matcher = matching_model.load_matching_model()
features = matching_model.build_pair_features(vectorizer, resume_texts, job_texts)
_, x_test, _, y_test = train_test_split(features, labels, test_size=0.2, random_state=42, stratify=labels)

match_probabilities = matcher.predict_proba(x_test)[:, 1]
match_predictions = (match_probabilities >= 0.5).astype(int)
print(classification_report(y_test, match_predictions, target_names=["No Fit", "Match"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y_test, match_probabilities, ax=axes[0], name="Matcher")
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set_title("ROC curve (higher and further left is better)")
ConfusionMatrixDisplay.from_predictions(
    y_test, match_predictions, display_labels=["No Fit", "Match"], ax=axes[1], colorbar=False
)
axes[1].set_title("Confusion matrix (threshold 0.5)")
plt.tight_layout()
plt.show()

## 3. Screening models (Logistic Regression for Hire/Reject + Random Forest for the AI score)

In [ ]:
frame = screening_model.load_screening_training_data()
features = frame[["profile_text", *screening_model.CATEGORICAL_COLUMNS, *screening_model.NUMERIC_COLUMNS]]
decisions = frame["decision_label"].astype(int)
_, x_test, _, y_test, _, score_test = train_test_split(
    features, decisions, frame["ai_score"].astype(float), test_size=0.2, random_state=42, stratify=decisions
)

decision_model, score_model = screening_model.load_screening_models()
decision_predictions = decision_model.predict(x_test)
print(classification_report(y_test, decision_predictions, target_names=["Reject", "Hire"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, decision_predictions, display_labels=["Reject", "Hire"], ax=axes[0], colorbar=False
)
axes[0].set_title("Hire/Reject - confusion matrix")

predicted_scores = score_model.predict(x_test).clip(0, 100)
axes[1].scatter(score_test, predicted_scores, alpha=0.5)
axes[1].plot([0, 100], [0, 100], "--", color="grey")
axes[1].set(xlabel="Actual AI score", ylabel="Predicted AI score", title="AI score regressor (dots on the line = perfect)")
plt.tight_layout()
plt.show()

## 4. The hybrid score on the demo resumes

Final score = 35% match + 25% skills + 25% AI screening score × skill match + 15% role relevance (only job-relevant fields count). The chart shows how much each "judge" adds for every sample resume against the Data Scientist job.

In [ ]:
job_text = (PROJECT_ROOT / "data" / "raw" / "job_descriptions" / "data_scientist.txt").read_text(encoding="utf-8")
rows = []
for path in sorted((PROJECT_ROOT / "data" / "samples" / "resumes").iterdir()):
    if path.suffix.lower() not in {".pdf", ".docx", ".txt"}:
        continue
    resume = extract_text_from_bytes(path.read_bytes(), path.name)
    result = recommend_as_dict(resume_text=resume.text, job_text=job_text)
    rows.append({"resume": path.name, "final_score": result["final_score"], "fit": result["fit_level"], **result["score_breakdown"]})

ranking = pd.DataFrame(rows).sort_values("final_score", ascending=False)
ranking

In [ ]:
components = ["match_probability_component", "skill_match_component", "screening_score_component", "role_confidence_component"]
ranking.set_index("resume")[components].plot.barh(stacked=True, figsize=(10, 5), title="How each judge contributes to the final score")
plt.xlabel("Points out of 100")
plt.gca().invert_yaxis()
plt.show()